# 07 — Attack simulation

**Purpose.** Byzantine (sign-flip) and backdoor attacks against undefended FedAvg.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/07_attack_simulation_*.png|pdf` and `reports/notebooks/07_attack_simulation.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "07_attack_simulation"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Sign-flip Byzantine clients

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
from federated_learning import attacks as A
d = make_dataset(n_samples=6000, seed=SEED); parts = partition("iid", d.y_train, 10, SEED); cfg = FLConfig(rounds=12, local_epochs=1, seed=SEED)
rows = []
for frac in (0, .1, .2, .3):
    mal = set(range(int(10 * frac)))
    r = run_fl(cfg, d, parts, client_hook=A.byzantine_hook("sign_flip", mal, 5.0) if mal else None); rows.append(dict(malicious_fraction=frac, acc=r.final["acc"]))
byz = pd.DataFrame(rows).round(4); display(byz); SUMMARY["byzantine"] = byz.to_dict("records")

,malicious_fraction,acc
0,0.0,0.7844
1,0.1,0.6639
2,0.2,0.3872
3,0.3,0.4006


## 2. Backdoor

In [3]:
rng = np.random.default_rng(SEED); mal = {0, 1, 2}
cd = {k: A.poison_client_data(d.X_train[parts[k]], d.y_train[parts[k]], .6, 0, rng)[:2] for k in mal}
ev = lambda m, w: A.backdoor_metrics(m, w, d.X_test, d.y_test, 0)
atk = run_fl(cfg, d, parts, client_data=cd, client_hook=A.boost_hook(mal, 3.0), eval_fn=ev); cln = run_fl(cfg, d, parts, eval_fn=ev)
print("attack success rate: attacked =", round(atk.final["asr"], 3), "| clean =", round(cln.final["asr"], 3)); SUMMARY["backdoor"] = dict(attacked_asr=atk.final["asr"], clean_asr=cln.final["asr"], attacked_acc=atk.final["acc"])

attack success rate: attacked = 1.0 | clean = 0.43


## 3. Figure

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(7, 2.7))
ax[0].plot(byz.malicious_fraction, byz.acc, "o-"); ax[0].set_xlabel("malicious fraction"); ax[0].set_ylabel("accuracy"); ax[0].set_title("Sign-flip (x5)")
ax[1].bar(["clean", "backdoored"], [cln.final["asr"], atk.final["asr"]], color=[PAL[0], PAL[1]]); ax[1].set_ylabel("attack success rate"); ax[1].set_title("Backdoor")
save_fig(fig, "attacks"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/07_attack_simulation.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.